# Low-Rank Approximations

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 08.04 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/05_low_rank_approximations.ipynb)

---

## 🎯 Learning Objective

Prove the Eckart-Young theorem's optimality guarantee, and learn randomized SVD and Nyström approximation for scaling up.

---

## 📐 Theory

[Notebook 01.07](../01_Linear_Algebra/07_SVD_and_PCA.ipynb) introduced truncated SVD and stated
that it gives the *best* rank-$k$ approximation. This notebook proves that claim is load-bearing
— not just "a good approximation," but provably optimal among *every* possible rank-$k$
matrix — and then covers the two standard tricks (`08.04`'s random matrices, reused as a tool
this time) for computing it when the matrix is too large to fully decompose.

### The Eckart-Young(-Mirsky) theorem

For $A\in\mathbb{R}^{m\times n}$ with SVD $A=U\Sigma V^\top$ and singular values
$\sigma_1\ge\sigma_2\ge\cdots\ge 0$, let $A_k = U_k\Sigma_k V_k^\top$ be the truncated SVD
keeping only the top $k$ terms. The **Eckart-Young theorem** (1936) states that $A_k$ is the
closest rank-$\le k$ matrix to $A$ in Frobenius norm, and gives the exact minimum error:

$$A_k = \arg\min_{\text{rank}(B)\le k} \|A - B\|_F, \qquad
\|A-A_k\|_F = \sqrt{\sum_{i>k}\sigma_i^2}$$

No other rank-$k$ matrix — however cleverly constructed — can beat this. Intuitively: $A_k$
keeps exactly the $k$ directions that capture the most of $A$'s "energy" ($\sum_i\sigma_i^2 =
\|A\|_F^2$, from [01.07](../01_Linear_Algebra/07_SVD_and_PCA.ipynb)), so anything you discard
from $A_k$ was necessarily the least important direction available, and any *different*
rank-$k$ matrix must be discarding at least as much. This notebook doesn't just state the
theorem — it hunts for a counterexample by constructing many alternative rank-$k$ matrices and
confirming none of them ever beats the truncated SVD.

### Randomized SVD

Computing a full SVD costs $O(mn\min(m,n))$ — expensive when $A$ is huge but you only want the
top few singular vectors. **Randomized SVD** (Halko, Martinsson & Tropp, 2011) sketches $A$
down to a small random subspace *first*, then runs an ordinary (cheap) SVD on the sketch:

1. Draw a random Gaussian matrix $\Omega\in\mathbb{R}^{n\times(k+p)}$ ($p$ = a few extra
   "oversampling" dimensions for safety), and form $Y = A\Omega \in \mathbb{R}^{m\times(k+p)}$ —
   $Y$'s columns span a subspace that, with high probability, captures nearly all of $A$'s
   top-$k$ range (this is the random-matrix-theory concentration idea from
   `08.04`, now used as a *tool* rather than an object of study).
2. Orthonormalize $Y$ via QR ([01.08](../01_Linear_Algebra/08_matrix_decompositions.ipynb)) to
   get $Q$.
3. Project $A$ down: $B = Q^\top A$ (small: $(k+p)\times n$), and run an exact SVD on $B$
   instead of $A$.
4. Lift the result back up: $U \approx QU_B$.

Because step 3's SVD acts on a much smaller matrix, the whole procedure is dramatically faster
whenever $k \ll \min(m,n)$, while still recovering (to very high accuracy) the leading singular
values/vectors that Eckart-Young says are the optimal ones to want.

### Nyström approximation

Randomized SVD sketches a general matrix. When $A$ is a symmetric PSD **kernel (Gram) matrix**
($A_{ij}=k(\mathbf{x}_i,\mathbf{x}_j)$, recalling [08.01](01_inner_product_spaces.ipynb)'s Gram
matrices), the **Nyström method** exploits that structure directly: pick $m\ll n$ "landmark"
points, form the small landmark-landmark block $W$ and the tall landmark-to-everyone block $C$,
and approximate

$$K \approx C\,W^{+}\,C^\top$$

where $W^{+}$ is the pseudoinverse of $W$. This never touches most of the $n\times n$ matrix at
all — only the $m$ chosen columns/rows — making it the standard way to scale kernel methods
(kernel SVMs, Gaussian processes) past the point where forming the full $n\times n$ Gram matrix
is feasible.

---

In [ ]:
# Setup
import numpy as np
from sklearn.utils.extmath import randomized_svd
import matplotlib.pyplot as plt
import time
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Eckart-Young says truncated SVD beats every other rank-$k$ matrix. Rather than trust that on
faith, we generate hundreds of alternative rank-3 matrices -- built by perturbing the SVD's own
answer by varying amounts, giving them every chance to occasionally get lucky -- and plot their
errors as a histogram against the truncated SVD's error as a single vertical line. If the
theorem is right, that line should sit at or before the very left edge of the histogram, with
nothing to its left.

In [ ]:
# Eckart-Young, put to the test: can ANY rank-3 matrix beat the truncated SVD?
rng = np.random.default_rng(0)
m, n, r = 10, 8, 3
A = rng.standard_normal((m, n))

U, S, Vt = np.linalg.svd(A, full_matrices=False)
A_svd = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
err_svd = np.linalg.norm(A - A_svd, "fro")

# Give the competitors every advantage: START from the SVD's own factors and perturb them by a
# RANDOM amount each time (not a single fixed perturbation size), so some trials search close to
# the optimum and others search far away.
n_competitors = 300
competitor_errors = []
for _ in range(n_competitors):
    perturbation_scale = rng.uniform(0.05, 1.0)
    B_competitor = U[:, :r] * S[:r] + perturbation_scale * rng.standard_normal((m, r))
    C_competitor = Vt[:r, :] + perturbation_scale * rng.standard_normal((r, n))
    competitor_errors.append(np.linalg.norm(A - B_competitor @ C_competitor, "fro"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(competitor_errors, bins=40, color="#4C72B0", alpha=0.75, label=f"{n_competitors} random rank-{r} competitors")
ax.axvline(err_svd, color="#C44E52", lw=2.5, label="truncated SVD (claimed optimum)")
ax.set_xlabel("Frobenius-norm reconstruction error")
ax.set_ylabel("count")
ax.set_title("Eckart-Young in action: nothing beats the truncated SVD")
ax.legend()
plt.tight_layout()
plt.show()

n_beating_svd = sum(e < err_svd for e in competitor_errors)
print(f"Truncated SVD error: {err_svd:.4f}")
print(f"Best (smallest) competitor error found: {min(competitor_errors):.4f}")
print(f"Competitors that beat the truncated SVD: {n_beating_svd}/{n_competitors}")

## 🐍 Implementation from Scratch

We stress-test Eckart-Young across 200 independently random matrices (not just one lucky
example), each time trying both pure-random and SVD-perturbed rank-$r$ competitors, and confirm
the exact closed-form error formula on every trial before moving on to the two scaling tricks:
randomized SVD (checked against exact SVD on a matrix large enough that the speedup is real)
and Nyström approximation (checked by how its error shrinks as more landmarks are used).

In [ ]:
rng = np.random.default_rng(0)

# --- Stress-test Eckart-Young across MANY random matrices, not just one ---
m, n, r = 9, 7, 2
n_matrices, n_competitors_per_matrix = 200, 20
matrices_where_svd_wins, formula_matches = 0, 0

for _ in range(n_matrices):
    A = rng.standard_normal((m, n))
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    A_svd = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
    err_svd = np.linalg.norm(A - A_svd, "fro")

    # Check the CLOSED-FORM error prediction independently of the reconstruction itself
    predicted_error = np.sqrt(np.sum(S[r:] ** 2))     # sqrt of sum of DISCARDED singular values squared
    if np.isclose(err_svd, predicted_error, atol=1e-8):
        formula_matches += 1

    beaten = False
    for _ in range(n_competitors_per_matrix):
        if rng.random() < 0.5:
            B_comp, C_comp = rng.standard_normal((m, r)), rng.standard_normal((r, n))   # pure random
        else:
            scale = rng.uniform(0.05, 1.0)                                              # perturbed SVD
            B_comp = U[:, :r] * S[:r] + scale * rng.standard_normal((m, r))
            C_comp = Vt[:r, :] + scale * rng.standard_normal((r, n))
        if np.linalg.norm(A - B_comp @ C_comp, "fro") < err_svd - 1e-9:
            beaten = True
    if not beaten:
        matrices_where_svd_wins += 1

print(f"SVD never beaten by any competitor: {matrices_where_svd_wins}/{n_matrices} matrices")
print(f"Closed-form error formula sqrt(sum of discarded sigma_i^2) matched exactly: {formula_matches}/{n_matrices}")

# --- Randomized SVD: same answer, dramatically faster on a matrix big enough to matter ---
print("\n--- Randomized SVD ---")
n_big = 800
U_true = np.linalg.qr(rng.standard_normal((n_big, 50)))[0]
V_true = np.linalg.qr(rng.standard_normal((n_big, 50)))[0]
S_true = np.exp(-np.arange(50) / 8.0) * 50                          # fast-decaying spectrum, like real data
A_big = U_true @ np.diag(S_true) @ V_true.T + 0.01 * rng.standard_normal((n_big, n_big))

k = 10
t0 = time.perf_counter()
U_full, S_full, Vt_full = np.linalg.svd(A_big, full_matrices=False)
time_full = time.perf_counter() - t0

t0 = time.perf_counter()
U_rand, S_rand, Vt_rand = randomized_svd(A_big, n_components=k, n_oversamples=10, n_iter=4, random_state=0)
time_rand = time.perf_counter() - t0

err_full_topk = np.linalg.norm(A_big - U_full[:, :k] @ np.diag(S_full[:k]) @ Vt_full[:k, :], "fro")
err_rand = np.linalg.norm(A_big - U_rand @ np.diag(S_rand) @ Vt_rand, "fro")
print(f"Matrix size {A_big.shape}, keeping top k={k} singular values/vectors")
print(f"  full SVD:      {time_full*1000:6.1f} ms, reconstruction error={err_full_topk:.6f}")
print(f"  randomized SVD:{time_rand*1000:6.1f} ms, reconstruction error={err_rand:.6f}  ({time_full/time_rand:.0f}x faster)")
print(f"  singular values match (top {k}):", np.allclose(S_full[:k], S_rand, atol=1e-3))

# --- Nystrom approximation of a kernel (Gram) matrix, error vs. number of landmarks ---
print("\n--- Nystrom approximation ---")
n_pts = 500
latent = rng.standard_normal((n_pts, 2))                              # low-dim latent structure
X_pts = latent @ rng.standard_normal((2, 5)) + 0.05 * rng.standard_normal((n_pts, 5))

def rbf_kernel(X, Y, gamma=0.5):
    sq_dists = ((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
    return np.exp(-gamma * sq_dists)

K_full = rbf_kernel(X_pts, X_pts)          # the true, full n x n Gram matrix (recall 08.01)

def nystrom_approx(X, n_landmarks, rng, gamma=0.5):
    idx = rng.choice(len(X), size=n_landmarks, replace=False)
    landmarks = X[idx]
    C = rbf_kernel(X, landmarks)                       # n x m: everyone vs. landmarks
    W_pinv = np.linalg.pinv(rbf_kernel(landmarks, landmarks))  # m x m landmark-landmark block
    return C @ W_pinv @ C.T

for n_landmarks in [10, 20, 40, 80]:
    K_nystrom = nystrom_approx(X_pts, n_landmarks, rng)
    rel_err = np.linalg.norm(K_full - K_nystrom, "fro") / np.linalg.norm(K_full, "fro")
    print(f"  {n_landmarks:3d} landmarks (of {n_pts}): relative Frobenius error = {rel_err:.4f}")

## 🤖 Why This Matters for AI

**Linformer** (Wang et al., 2020) speeds up self-attention from $O(L^2)$ to $O(L)$ in sequence
length $L$ by projecting the keys and values down to a much shorter length before computing
attention — a low-rank approximation of the attention mechanism itself, justified by the
empirical observation that trained attention matrices are approximately low-rank. We check the
premise directly: does an attention matrix built from **structured** (correlated) queries/keys
have a faster-decaying singular value spectrum than one built from unstructured **random**
queries/keys? If Eckart-Young-style low-rank approximation is going to work well on attention,
the structured case needs to show it can be truncated with far less error — and this is also
exactly the "sum of the top singular values captures most of the update" argument that
**LoRA** (next notebook, `08.06`) turns into a fine-tuning method rather than a compression
trick applied after training.

In [ ]:
# Linformer's premise, checked directly: is a STRUCTURED attention matrix more compressible
# (faster singular-value decay) than an UNSTRUCTURED one? Eckart-Young says truncated SVD is
# the best possible rank-k approximation either way -- the real question is how good "best"
# actually gets, which depends entirely on how much structure is there to exploit.
# A FRESH, independently-seeded RNG here (not the notebook's shared `rng`, already advanced by
# 200+ draws above) keeps this self-contained demo reproducible on its own, and we average over
# many independent trials rather than trusting a single lucky/unlucky draw.
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

seq_len, d_k = 200, 16
ranks_to_check = [2, 5, 10, 20, 40]
n_trials = 30
latent_dim = 2   # queries/keys built from only 2 shared latent directions -- strong structure

structured_errors = {k: [] for k in ranks_to_check}
random_errors = {k: [] for k in ranks_to_check}

rng_attn = np.random.default_rng(123)
for _ in range(n_trials):
    # Structured case: queries/keys share a low-dimensional latent factor (like real trained
    # attention, which tends to attend to a limited number of "types" of context).
    Q_latent, K_latent = rng_attn.standard_normal((seq_len, latent_dim)), rng_attn.standard_normal((seq_len, latent_dim))
    proj_up = rng_attn.standard_normal((latent_dim, d_k))
    Q_structured = Q_latent @ proj_up + 0.15 * rng_attn.standard_normal((seq_len, d_k))
    K_structured = K_latent @ proj_up + 0.15 * rng_attn.standard_normal((seq_len, d_k))
    P_structured = softmax(Q_structured @ K_structured.T / np.sqrt(d_k), axis=-1)

    # Control: queries/keys with NO shared structure at all.
    Q_random, K_random = rng_attn.standard_normal((seq_len, d_k)), rng_attn.standard_normal((seq_len, d_k))
    P_random = softmax(Q_random @ K_random.T / np.sqrt(d_k), axis=-1)

    Us, Ss, Vts = np.linalg.svd(P_structured, full_matrices=False)
    Ur, Sr, Vtr = np.linalg.svd(P_random, full_matrices=False)
    for k in ranks_to_check:
        structured_errors[k].append(
            np.linalg.norm(P_structured - Us[:, :k] @ np.diag(Ss[:k]) @ Vts[:k, :], "fro") / np.linalg.norm(P_structured, "fro"))
        random_errors[k].append(
            np.linalg.norm(P_random - Ur[:, :k] @ np.diag(Sr[:k]) @ Vtr[:k, :], "fro") / np.linalg.norm(P_random, "fro"))

print(f"Averaged over {n_trials} independent trials each:")
print(f"{'rank k':>8} {'mean structured error':>24} {'mean random error':>20}")
for k in ranks_to_check:
    mean_structured = np.mean(structured_errors[k])
    mean_random = np.mean(random_errors[k])
    print(f"{k:>8} {mean_structured:>24.4f} {mean_random:>20.4f}")

print("\nAt every rank, the structured attention matrix truncates with LESS error, on average,")
print("than the unstructured control -- confirming Linformer's premise: attention matrices with")
print("genuine structure (as trained attention has) are the cases where a low-rank projection of")
print("K, V loses the least. This is the same 'low-rank approximation quality depends on real")
print("structure, not just on Eckart-Young being true' idea that LoRA (08.06) relies on for")
print("weight UPDATES rather than attention matrices.")

## 🏋️ Exercises

### Warm-up
1. For $A=\begin{pmatrix}4&0\\0&1\end{pmatrix}$, its singular values are $4$ and $1$ (it's
   already diagonal). Compute the best rank-1 approximation $A_1$ by hand (keep only the larger
   singular value), then compute $\|A-A_1\|_F$ by hand and confirm it equals
   $\sqrt{\sum_{i>1}\sigma_i^2}=1$. Verify both with `np.linalg.svd`.

### Practice
2. Repeat the "hunt for a counterexample" stress test with a rank $r=1$ instead of $r=2$, and
   with a *larger* competitor pool (1000 competitors per matrix instead of 20). Confirm the
   truncated SVD is still never beaten. Then try to explain in a sentence why making $r$
   *smaller* relative to $\min(m,n)$ doesn't change whether Eckart-Young holds, even though it
   changes how much error remains.

### Challenge
3. In the randomized SVD cell, shrink `n_oversamples` from 10 down to 0 and `n_iter` from 4 down
   to 0, and re-run the comparison against exact SVD on the same matrix. Report how much the
   singular-value accuracy degrades. Then explain, using this notebook's Nyström section as a
   reference point, why BOTH randomized SVD and Nyström approximation trade a small, controllable
   amount of accuracy for a large speedup, but do so via genuinely different mechanisms (random
   projection vs. a fixed subset of "landmark" rows/columns).

---

## 📚 Further Reading
- Eckart & Young, ["The Approximation of One Matrix by Another of Lower Rank"](https://link.springer.com/article/10.1007/BF02288367), Psychometrika (1936)
- Halko, Martinsson & Tropp, ["Finding Structure with Randomness"](https://arxiv.org/abs/0909.4061)
- Williams & Seeger, ["Using the Nyström Method to Speed Up Kernel Machines"](https://papers.neurips.cc/paper/1866-using-the-nystrom-method-to-speed-up-kernel-machines) (NeurIPS 2000)
- Wang et al., ["Linformer: Self-Attention with Linear Complexity"](https://arxiv.org/abs/2006.04768)

---

*Next notebook: [LoRA and Efficient Fine-Tuning](06_lora_and_efficient_finetuning.ipynb)*